In [14]:
!pwd

/home/prent/Repos/One-footed-bride-tuning


In [15]:
import logging, os, sys, time
import json
import numpy as np
from math import exp
from importlib import reload
from collections import Counter, defaultdict
user = 'prent'
base_dir = os.path.join('/home', user, 'Repos', 'One-footed-bride-tuning') 
WAVE_DIR = os.path.join(base_dir, 'Music', 'sflib')
numpy_dir = os.path.join(base_dir, 'Archive')
for i in range(len(sys.path)):
    print(f"Path {i}: {sys.path[i]}")
# point it to the base_dir to import the modules:
sys.path.append(base_dir)
import diamond_music_utils as dmu
import adaptive_tuning_util as atu 
from itertools import count, combinations, permutations
rng = np.random.default_rng()

np.set_printoptions(legacy='1.25')
import logging


Path 0: /home/prent/miniforge3/envs/csound/lib/python313.zip
Path 1: /home/prent/miniforge3/envs/csound/lib/python3.13
Path 2: /home/prent/miniforge3/envs/csound/lib/python3.13/lib-dynload
Path 3: 
Path 4: /home/prent/miniforge3/envs/csound/lib/python3.13/site-packages
Path 5: /home/prent/Repos/One-footed-bride-tuning


In [16]:
def start_logger(logfile: str, level=logging.INFO):
    # Reset the root logger
    root = logging.getLogger()
    root.handlers.clear()
    root.setLevel(level)

    # File handler only
    fh = logging.FileHandler(logfile, mode='w')
    fh.setLevel(level)

    formatter = logging.Formatter(
        fmt='%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%d'
    )
    fh.setFormatter(formatter)

    root.addHandler(fh)

    root.info(f"Logger started, writing to {logfile}, level={logging.getLevelName(level)}")
    return root


In [17]:
def compare_chord_files(original_file=None, transformed_file=None, tolerance=5, worst_n = None, \
        print_items=True, worse_than=None, method=combinations, print_pitch_class_gaps=False, limit_max=31, max_gap=30, only_changes=False):
    
    tonal_diamond = atu.build_tonal_diamond(limit_max)[:-1]
    chord_scorer = atu.ChordScorer(tonal_diamond)
    low_number_ratios = atu.LowNumberRatioIntervals(tonal_diamond)
    keys = atu.set_accidentals(False) # returns sharps. For flats, use set_accidentals(True)

    def build_gap_slots(prev_vals, curr_vals):
        if prev_vals is None:
            return ['    ' for _ in range(4)], []
        prev_pc = atu.pitch_class_from_cents(prev_vals)
        curr_pc = atu.pitch_class_from_cents(curr_vals)
        slots = []
        records = []
        for voice_idx, (pc, curr_cent) in enumerate(zip(curr_pc, curr_vals)):
            prev_pc_voice = prev_pc[voice_idx]
            if prev_pc_voice != pc:
                slots.append('    ')
                continue
            prev_cent = prev_vals[voice_idx]
            gap = atu.wrap600(curr_cent - prev_cent)
            if abs(gap) > max_gap:
                slots.append(f'{gap:+4.0f}')
                records.append((abs(gap), pc, gap, voice_idx))
            else:
                slots.append('    ')
        return slots, records
    
    if original_file is not None:
        filename = os.path.join(original_file)
        mtime = time.ctime(os.path.getmtime(filename))
        final_cent_value_chorale = np.load(filename).T
    else:
        final_cent_value_chorale = np.zeros_like((1, 4))
        mtime = None
    if transformed_file is not None:
        filename = os.path.join(transformed_file)
        mtime2 = time.ctime(os.path.getmtime(filename))
        adjusted_cent_value_chorale = np.load(filename).T
        
    else: 
        adjusted_cent_value_chorale = np.zeros_like(final_cent_value_chorale)
        mtime2 = None
        
    cent_value_chord_prev = np.zeros_like(final_cent_value_chorale[0])
    final_score = np.zeros(final_cent_value_chorale.shape[0])
    final_score_adjusted = np.zeros(final_cent_value_chorale.shape[0])
    prev_adjusted_cent_values = None
    prev_final_cent_values = None
    prev_inx = None
    all_gap_records = []
    print(f'first file: {original_file} saved @ {mtime}\nsecond file: {transformed_file} saved @ {mtime2}')
    print(f'#    original cent values  score  transformed cents      score          delta      🗸 pc  note names')
    for inx, cent_value_chord_current, adjusted_cent_value_chord in zip(count(0,1), final_cent_value_chorale, adjusted_cent_value_chorale):
        cent_str = atu.format_chord(cent_value_chord_current,4)
        adj_cent_str = atu.format_chord(adjusted_cent_value_chord,4)
        display_score = chord_scorer.score_chord(cent_value_chord_current, tolerance=tolerance)
        display_score_adjusted = chord_scorer.score_chord(adjusted_cent_value_chord, tolerance=tolerance)
        display_score_str = f'{display_score:>6}'
        display_score_adjusted_str = f'{display_score_adjusted:>6}'
        delta_cents = atu.wrap600(adjusted_cent_value_chord - cent_value_chord_current)
        delta_str = atu.format_chord(delta_cents,4)
        pitch_class_equal = np.array_equal(atu.pitch_class_from_cents(cent_value_chord_current),atu.pitch_class_from_cents(adjusted_cent_value_chord))
        pitch_class_str = str(keys[atu.pitch_class_from_cents(cent_value_chord_current)])
        # remove the braces and quote marks from the pitch class string for cleaner display
        pitch_class_str = pitch_class_str.replace('[', '').replace(']', '').replace("'", "")
        # Use fixed-width spaces instead of tabs for proper alignment
        line = (
            f'{inx:>3}: {cent_str}  {display_score_str}  '
            f'{adj_cent_str}  {display_score_adjusted_str}  {delta_str}  {pitch_class_equal} {pitch_class_str}'
        )
        if not np.array_equal(cent_value_chord_prev, cent_value_chord_current):
            # Check if only_changes is enabled and if there are actual changes
            has_changes = not np.array_equal(cent_value_chord_current, adjusted_cent_value_chord)
            should_print = (not only_changes) or has_changes
            
            if print_items and should_print:
                print(line)
                if print_pitch_class_gaps:
                    final_slots, final_records = build_gap_slots(prev_final_cent_values, cent_value_chord_current)
                    adjusted_slots, adjusted_records = build_gap_slots(prev_adjusted_cent_values, adjusted_cent_value_chord)
                    # Calculate lead spacing to align gap slots under the cent values
                    # lead_original: 5 chars for "{inx:>3}: " (e.g. " 68: ")
                    lead_original = 5
                    # lead_adjusted: prefix + cent_str + 2 spaces + score + 2 spaces
                    # = 5 + 19 + 2 + 6 + 2 = 34 chars to start of adj_cent_str
                    lead_adjusted = 34
                    if any(slot.strip() for slot in final_slots):
                        final_context = ''
                        if final_records and prev_inx is not None:
                            details = ', '.join(
                                f"v{voice_idx}PC{pc}:{gap:+.0f}"
                                for (_, pc, gap, voice_idx) in final_records
                            )
                            final_context = f'  ({prev_inx}->{inx} {details})'
                        print(' ' * lead_original + ' '.join(final_slots) + final_context)
                    if any(slot.strip() for slot in adjusted_slots):
                        adjusted_context = ''
                        if adjusted_records and prev_inx is not None:
                            details = ', '.join(
                                f"v{voice_idx}PC{pc}:{gap:+.0f}"
                                for (_, pc, gap, voice_idx) in adjusted_records
                            )
                            adjusted_context = f'  ({prev_inx}->{inx} {details})'
                        print(' ' * lead_adjusted + ' '.join(adjusted_slots) + adjusted_context)
                    all_gap_records.extend([(rec[0], prev_inx, inx, 'final', rec[1], rec[2], rec[3]) for rec in final_records])
                    all_gap_records.extend([(rec[0], prev_inx, inx, 'adjusted', rec[1], rec[2], rec[3]) for rec in adjusted_records])
            prev_inx = inx
        final_score[inx] = chord_scorer.score_chord(cent_value_chord_current, tolerance=tolerance, method=method)
        final_score_adjusted[inx] = chord_scorer.score_chord(adjusted_cent_value_chord, tolerance=tolerance, method=method)
        prev_adjusted_cent_values = adjusted_cent_value_chord.copy()
        prev_final_cent_values = cent_value_chord_current.copy()
    
    print(f'Scores for {original_file}')
    print(
        f"mean: {np.round(np.mean(final_score),1)}, ",
        f"median: {np.median(final_score)}, ",
        f"min: {np.min(final_score)}, ",
        f"max: {np.max(final_score)}, ",
        f"argmax: {np.argmax(final_score)}, ",
        f"deciles: {np.percentile(final_score, np.arange(0, 100, 10))}")
    print(f'Scores for {transformed_file}')
    print(
        f"mean: {np.round(np.mean(final_score_adjusted),1)}, ",
        f"median: {np.median(final_score_adjusted)}, ",
        f"min: {np.min(final_score_adjusted)}, ",
        f"max: {np.max(final_score_adjusted)}, ",
        f"argmax: {np.argmax(final_score_adjusted)}, ",
        f"deciles: {np.percentile(final_score_adjusted, np.arange(0, 100, 10))}"
    )
    print()
    if worst_n != None or worse_than != None:
        if worst_n != None: print(f'Worst {worst_n} chords:')
        elif worse_than != None: print(f'Worse than {worse_than}')
        
        print(f'{original_file}')
        print(f'#\tCent values\t\tscore')
        sort_idx = np.argsort(final_score)[::-1]
        sorted_scores = final_score[sort_idx]         
        cent_value_chorale = final_cent_value_chorale[sort_idx]
        cent_value_chord_prev = np.zeros_like(final_cent_value_chorale[0])
        for inx, chord_num, cent_value_chord_current, score in zip(count(0,1), sort_idx, cent_value_chorale, sorted_scores):
            if worst_n != None and inx >= worst_n: break
            if worse_than != None and score < worse_than: break 
            if not np.array_equal(cent_value_chord_prev, cent_value_chord_current):
                print(f'{chord_num:>3}: {atu.format_chord(cent_value_chord_current,4)}\t{score:>6}')
        print()
        print(f'{transformed_file}')
        print(f'#\tCent values\t\tscore')
        sort_idx = np.argsort(final_score_adjusted)[::-1]
        sorted_scores = final_score_adjusted[sort_idx]         
        cent_value_chorale = adjusted_cent_value_chorale[sort_idx]
        cent_value_chord_prev = np.zeros_like(final_cent_value_chorale[0])
        for inx, chord_num, cent_value_chord_current, score in zip(count(0,1), sort_idx, adjusted_cent_value_chorale, sorted_scores):
            if worst_n != None and inx >= worst_n: break
            if worse_than != None and score < worse_than: break 
            if not np.array_equal(cent_value_chord_prev, cent_value_chord_current):
                print(f'{chord_num:>3}: {atu.format_chord(cent_value_chord_current,4)}\t{score:>6}')
    
    if print_pitch_class_gaps and all_gap_records:
        print("\nLargest gaps (sorted by absolute value, descending):")
        sorted_gaps = sorted(all_gap_records, key=lambda x: x[0], reverse=True)
        for abs_gap, prev_idx, curr_idx, label, pc, gap, voice_idx in sorted_gaps[:20]:
            sign = "+" if gap > 0 else ""
            print(f"{label[0].upper()} {prev_idx}->{curr_idx} voice {voice_idx} PC{pc}: {sign}{gap:.1f}")
    print()


In [24]:
# compare chords matching these files created on 3/3/26
# Archive/opt/tolerance-3/bwv254-sa-opt.npy
# Archive/opt/tolerance-3/bwv257-trans-sa-opt.npy
# Archive/straw-man/t1_r1.25_s0_md33_sn0_lm17/bwv264-trans-sa-opt.npy
tolerance=1
max_delta=33
print(f'{numpy_dir = }')
sub_dir = 'straw-man-h20'
start_logger('compare_chord_files.log', level=logging.INFO)
for tolerance in [1]: #,2,3,4]:
    for chorale_num in ['253']: # np.arange(253, 265): 
        prefix = ''
        original_file = os.path.join(numpy_dir, sub_dir, prefix, f'bwv{chorale_num}-opt.npy') # original_file
        transformed_file = os.path.join(numpy_dir, sub_dir, prefix, f'bwv{chorale_num}-trans-sa-opt.npy') # transformed_file
        # print(f'{original_file} vs {transformed_file}')
        compare_chord_files(original_file = original_file, transformed_file=transformed_file, limit_max=17, tolerance=1, worst_n=10, print_items=True, print_pitch_class_gaps=True, only_changes=False)  


numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive'
first file: /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man-h20/bwv253-opt.npy saved @ Sat Mar 28 12:43:18 2026
second file: /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man-h20/bwv253-trans-sa-opt.npy saved @ Sat Mar 28 12:46:27 2026
#    original cent values  score  transformed cents      score          delta      🗸 pc  note names
  0:  113  927  429  927    43.0   113  927  429  927    43.0     0    0    0    0  True C♯ A♮ E♮ A♮
  1:  113  927  429  927    43.0   113  927  429  927    43.0     0    0    0    0  True C♯ A♮ E♮ A♮
  2:  113  927  429  927    43.0   113  927  429  927    43.0     0    0    0    0  True C♯ A♮ E♮ A♮
  3:  113  927  429  927    43.0   113  927  429  927    43.0     0    0    0    0  True C♯ A♮ E♮ A♮
  4:  101  915  417  915    43.0   113  927  429  927    43.0    12   12   12   12  True C♯ A♮ E♮ A♮
  5:  101  915  417  915    43.0   113  927  429  927    43.0    12   12 

In [23]:
test_array = np.array([441, 1143, 441, 757])
for limit_max in [17, 19, 23, 29, 31]:
    tonal_diamond = atu.build_tonal_diamond(limit_max)
    chord_scorer = atu.ChordScorer(tonal_diamond)
    low_number_ratios = atu.LowNumberRatioIntervals(tonal_diamond)
    print(f'{limit_max = }, {chord_scorer.score_chord(test_array, tolerance=1) = }')

limit_max = 17, chord_scorer.score_chord(test_array, tolerance=1) = 45.0
limit_max = 19, chord_scorer.score_chord(test_array, tolerance=1) = 45.0
limit_max = 23, chord_scorer.score_chord(test_array, tolerance=1) = 45.0
limit_max = 29, chord_scorer.score_chord(test_array, tolerance=1) = 45.0
limit_max = 31, chord_scorer.score_chord(test_array, tolerance=1) = 45.0


In [ ]:
# t1_r1.50_s0_md33_sn0
tolerance=1
max_delta=33
start_logger('compare_chord_files.log', level=logging.INFO)
for ratio_factor in [1.50]:
    for stability_factor in ["0"]:
        for snap_tolerance in [0]:
            for chorale_num in [257]:  # np.arange(253, 265): # [261]: 
                prefix = f't{tolerance}_r{ratio_factor:.2f}_s{stability_factor}_md{max_delta}_sn{snap_tolerance}'
                original_file = os.path.join(numpy_dir, prefix, f'bwv{chorale_num}-opt.npy') # original_file

                # Archive/straw-man/t1_r1.75_s1.25_md28_sn10/bwv253-trans-sa-opt.npy
                transformed_file = os.path.join(numpy_dir, prefix, f'bwv{chorale_num}-trans-sa-opt.npy') # transformed_file
                print(f'{original_file} vs {transformed_file}')
                compare_chord_files(original_file = original_file, transformed_file=transformed_file, limit_max=19, tolerance=1, worst_n=10, print_items=True, print_pitch_class_gaps=True, only_changes=False)  


/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t1_r1.50_s0_md33_sn0/bwv257-opt.npy vs /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t1_r1.50_s0_md33_sn0/bwv257-trans-sa-opt.npy
first file: /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t1_r1.50_s0_md33_sn0/bwv257-opt.npy saved @ Wed Feb 25 15:56:01 2026
second file: /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t1_r1.50_s0_md33_sn0/bwv257-trans-sa-opt.npy saved @ Wed Feb 25 16:14:09 2026
#    original cent values  score  transformed cents      score          delta      🗸 pc  note names
  0: 1196  880  382  880    45.0  1196  880  382  880    45.0     0    0    0    0  True C♮ A♮ E♮ A♮
  1: 1196  880  382  880    45.0  1196  880  382  880    45.0     0    0    0    0  True C♮ A♮ E♮ A♮
  2: 1196  880  382  880    45.0  1196  880  382  880    45.0     0    0    0    0  True C♮ A♮ E♮ A♮
  3: 1196  880  382  880    45.0  1196  880  382  880    45.0     0    0    0    0  True C♮ A♮ E♮ A♮
